In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# AWG Initialisation

In [ ]:
#Jupyter cell 1 — imports and connection
from rigol_dg1022 import RigolDG1022

RESOURCE = "USB0::0x1AB1::0x0642::DG1ZA231701902::INSTR"  # update if your VISA address differs
gen = RigolDG1022(RESOURCE)  # auto_open=True by default
print("Connected to:", gen.idn)

## Set Wavefrom

In [ ]:
# Sine wave on CH1, 2 Vpp, 0 V offset
freq=int(1e6)

Vpi=5.97
coeff=.8
Vpp_Ch1=round(Vpi*coeff,3)
#Set waveform on CH1
gen.set_waveform(ch=1, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch1, offset_v=0.0)

#Set phase of CH1
gen.set_phase_deg(ch=1, phase_deg=0) 

#Set ch1 impedance to 50 Ohms or INF
gen.set_load(ch=1, load='50')




# PhasemeterClient — quick start

In [ ]:
from phasemeter_client_v2 import PhasemeterClient


# Fill in the target of your Moku (e.g., '192.168.1.100' or link-local IPv6)
TARGET = '[fe80::7269:79ff:feb7:d15%6]'  # <-- replace if needed
# client = PhasemeterClient(
#     target=TARGET,
#     phase_units='cycles',   # or 'deg' depending on API
#     wrap_output=True,
#     input_range='1Vpp',
#     impedance='50Ohm',
#     coupling='DC',
#     pll_bandwidth='1kHz',
#     poll_sec=120,
# )

client = PhasemeterClient(target=TARGET, poll_sec=0.1)
client.load(f0_hz=1e6)  # optional initial lock

## Change device settings on the fly

In [ ]:
# Change front-end input range for channel 1, then both
client.set_frontend(1, input_range='1Vpp')
client.set_frontend(2, input_range='1Vpp')
# Change PLL bandwidth for both channels
client.set_bandwidth_all('1kHz')
# Move center frequency (both channels track the same f)
client.set_frequency_all(1e6)

# Laser Setup

In [ ]:
import TLX_5

import time

laser=TLX_5.TRL_5()

port="COM4"

laser.connect(port)

## Set Laser Wavelength (nm)

In [ ]:
wl="1560.0"

laser.change_WL(wl)  # in nm

laser.laser_ON()


### Laser OFF

In [ ]:
laser.laser_OFF()

# Test 

In [ ]:
# Set Sine wave frequency
f=5e6

freq=int(f)
gen.set_waveform(ch=1, wave="SIN", freq_hz=freq, ampl_vpp=Vpp_Ch1, offset_v=0.0)


In [ ]:
# Move center frequency (both channels track the same f)
client.set_frequency_all(f)


In [ ]:
#Turn Ch1 ouput ON
gen.output_on(1)

time.sleep(.5)

#dphi_deg, frame = client.read_phase_difference_deg()

client.reset_unwrap()

dphi_deg, frame=client.read_delta_phi_deg(mode='unwrapped')

print('Δφ [deg]:', dphi_deg)
print('f1 [Hz]:', frame['ch1']['frequency'], 'f2 [Hz]:', frame['ch2']['frequency'])

#laser.laser_OFF()

gen.output_off(1)

In [ ]:
def mesurement_avg(n_av:int):
    ph_diff=[]
    f1=[]
    f2=[]
    time.sleep(.55)
    i=0
    while i<n_av:
       dphi_deg, frame=client.read_delta_phi_deg(mode='unwrapped')
       ph_diff.append(dphi_deg)
       f1.append( frame['ch1']['frequency'])
       f2.append( frame['ch2']['frequency'])
       i+=1
    f1_avg=np.mean(f1)
    f2_avg=np.mean(f2)
    ph_diff_avg=np.mean(ph_diff)
    ph_diff_std=np.std(ph_diff)
    return f1_avg,f2_avg,ph_diff_avg,ph_diff_std
        

In [ ]:
gen.output_on(1)

results=mesurement_avg(10)

gen.output_off(1)

print(f'f1_avg={results[0]} Hz; f2_avg={results[1]} Hz; ph_diff_avg={results[2]} ± {results[3]}\n')




In [ ]:
def f_sweep(n_av:int, freq):
    size=len(freq)
    i=0
    f1=[]
    f2=[]
    phase=[]
    gen.output_on(1)
    while i<size:
        f=int(freq[i])
        gen.set_waveform(ch=1, wave="SIN", freq_hz=f, ampl_vpp=Vpp_Ch1, offset_v=0.0)
        client.set_frequency_all(f)
        results=mesurement_avg(20)
        print(f'f1_avg={results[0]} Hz; f2_avg={results[1]} Hz; ph_diff_avg={results[2]} ± {results[3]}\n')
        f1.append(results[0])
        f2.append(results[1])
        phase.append(results[2])
        i+=1
    gen.output_off(1)
        
    return(f1,f2,phase)
        
        
        
        


In [ ]:
freqs=np.linspace(5e6,25e6,20)

#freqs=np.linspace(6e6,22e6,51)

n=10
client.reset_unwrap()
res1=f_sweep(n,freqs)

In [ ]:
client.reset_unwrap()
res2=f_sweep(n,freqs)

In [ ]:
ff=[]
i=0
while i<len(freqs):
    delta=res2[2][i]-res1[2][i]
    print(f'Calibration={delta}\n')
    ff.append(delta)
    
    i+=1

In [ ]:
ph_diff_cab=[]
i=0
while i<len(freqs):
    delta=(res2[2][i]-res1[2][i])
    ph_diff_cab.append(delta)
    i+=1

In [ ]:
f_Ch1=res1[0]

#phase_diff=ph_diff_cab

phase_diff=res1[2]


fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    f_Ch1,phase_diff,
    marker='o',         # square markers
    markersize=3,
    markerfacecolor='red',
    markeredgecolor='black',
    linestyle='None'    # disables line
)

# ax.plot(
#     f_Ch1,res1[2],
#     marker='o',         # square markers
#     markersize=3,
#     markerfacecolor='red',
#     markeredgecolor='black',
#     linestyle='None'    # disables line
# )
    
ax.set_xlim(np.min(f_Ch1),np.max(f_Ch1))
ax.set_ylim(0,np.max(phase_diff)+0.1*np.max(phase_diff))
#plt.title(f"Power Stability\n Mean={mean:.2f} Std={std:.2f}")
plt.show()

In [ ]:
laser.laser_OFF()

In [ ]:
client.close()

In [ ]:
gen.close()

In [ ]:
freqs=np.linspace(5e6,25e6,101)
gen.output_off(1)

def freq_generator():
    for f in freqs:
        gen.set_waveform(ch=1, wave="SIN", freq_hz=f, ampl_vpp=Vpp_Ch1, offset_v=0.0)
        yield float(f)


client.reset_unwrap()
gen.output_on(1)
rows = client.sweep_phase_vs_frequency_external(
    freq_generator(),
    settle_s=5,
    samples_per_point=30,
    mode="both",                 # wrapped/unwrapped/raw all recorded
    set_pll_to_frequency=True,   # recommended: keep PLL centered on AWG freq
    reacquire_each_step=True,
    reset_unwrap=False
)


gen.output_off(1)

In [ ]:
rows

In [ ]:
client.plot_sweep(rows, y_key="delta_phi_unwrapped_deg", title="Unwrapped Δφ vs Frequency")